# Weather Dataset Cleaning and Analysis (Agricultural Focus)

This notebook demonstrates the end-to-end process of fetching historical weather data from the **Open-Meteo API**, cleaning it using **Pandas**, and preparing it for agricultural machine learning tasks.

### Dataset Information
*   **Source:** Open-Meteo Historical Weather API (ERA5 reanalysis data).
*   **Location:** Solapur, Maharashtra (Latitude: 17.6715, Longitude: 75.9011).
*   **Time Period:** 2023-01-01 to 2023-12-31.
*   **Expanded Agricultural Variables:**
    *   `temperature_2m`, `relative_humidity_2m`, `precipitation`
    *   `dewpoint_2m`, `surface_pressure`, `cloud_cover`
    *   `wind_speed_10m`, `wind_direction_10m`
    *   `soil_temperature_0_to_7cm`, `soil_moisture_0_to_7cm`, `soil_moisture_7_to_28cm`
    *   `shortwave_radiation`, `evapotranspiration`, `vapour_pressure_deficit` (VPD)
*   **Frequency:** Hourly intervals.
*   **Goal:** Create a robust time-series dataset including soil, wind, and radiation metrics to predict crop stress and irrigation needs.

### 1. Install Libraries: requests and pandas
To communicate with the Open-Meteo API and process the tabular data, you need to ensure the appropriate Python libraries are available. `requests` handles the HTTP GET request to the API, while `pandas` provides the powerful DataFrame structure to clean and analyze the information.

In [3]:
# While requests and pandas are usually pre-installed in Colab, this ensures they are available
import requests
import pandas as pd

### 2. Configure the API Parameters: Endpoint, location, dates, and variables
Open-Meteo uses specific endpoints for different types of data. For historical records from the archive, we use the `archive-api` endpoint. We will configure the latitude and longitude for Solapur and specify the 2023 date range.

In [4]:
# Updated configuration with 14 agricultural features
url = "https://archive-api.open-meteo.com/v1/archive"

params = {
    "latitude": 17.6715,
    "longitude": 75.9011,
    "start_date": "2023-01-01",
    "end_date": "2023-12-31",
    "hourly": [
        "temperature_2m",
        "relative_humidity_2m",
        "precipitation",
        "dewpoint_2m",
        "surface_pressure",
        "cloud_cover",
        "wind_speed_10m",
        "wind_direction_10m",
        "soil_temperature_0_to_7cm",
        "soil_moisture_0_to_7cm",
        "soil_moisture_7_to_28cm",
        "shortwave_radiation",
        "evapotranspiration",
        "vapour_pressure_deficit"
    ],
    "timezone": "auto"
}

### 3. Fetch the Data: Request and convert JSON to DataFrame
Using the `requests` library, we send the GET request. If the status code is 200 (Success), we extract the nested `hourly` data and load it into a Pandas DataFrame.

In [5]:
import os

print("Fetching extended agricultural weather data...")
response = requests.get(url, params=params)

if response.status_code == 200:
    data = response.json()
    df = pd.DataFrame(data['hourly'])

    # Save raw data
    raw_path = r'E:\Apollo_AgriVerse\02_Datasets\Raw\weather'
    os.makedirs(raw_path, exist_ok=True)
    df.to_csv(os.path.join(raw_path, "weather_raw.csv"), index=False)

    print(f"Success! Data shape: {df.shape}")
    display(df.head())
else:
    print(f"Error: {response.status_code}")

Fetching extended agricultural weather data...
Success! Data shape: (8760, 15)


,time,temperature_2m,relative_humidity_2m,precipitation,dewpoint_2m,surface_pressure,cloud_cover,wind_speed_10m,wind_direction_10m,soil_temperature_0_to_7cm,soil_moisture_0_to_7cm,soil_moisture_7_to_28cm,shortwave_radiation,evapotranspiration,vapour_pressure_deficit
0,2023-01-01T00:00,23.8,54,0.0,14.0,962.5,73,3.8,41,26.0,0.044,0.311,0.0,None,1.35
1,2023-01-01T01:00,22.8,54,0.0,13.0,962.0,98,9.8,73,25.1,0.044,0.311,0.0,None,1.27
2,2023-01-01T02:00,21.7,59,0.0,13.3,961.5,96,8.1,58,24.3,0.044,0.311,0.0,None,1.07
3,2023-01-01T03:00,21.5,60,0.0,13.5,961.5,96,2.5,315,23.4,0.044,0.311,0.0,None,1.01
4,2023-01-01T04:00,20.4,66,0.0,14.0,961.5,95,3.7,259,22.7,0.044,0.311,0.0,None,0.80


### 4. Clean and Format the Data: Format dates and rename columns
We now convert the 'time' column from strings to Python `datetime` objects and rename the columns to more intuitive names like `temperature_c` and `humidity_percent`.

In [6]:
# 1. Convert 'time' to Datetime objects
df['time'] = pd.to_datetime(df['time'])

# 2. Rename columns to standardized agricultural terms
df.rename(columns={
    'temperature_2m': 'temperature_c',
    'relative_humidity_2m': 'humidity_percent',
    'precipitation': 'precipitation_mm',
    'dewpoint_2m': 'dewpoint_c',
    'surface_pressure': 'pressure_hpa',
    'cloud_cover': 'cloud_cover_pct',
    'wind_speed_10m': 'wind_speed_kmh',
    'wind_direction_10m': 'wind_direction_deg',
    'soil_temperature_0_to_7cm': 'soil_temp_shallow_c',
    'soil_moisture_0_to_7cm': 'soil_moisture_shallow_pct',
    'soil_moisture_7_to_28cm': 'soil_moisture_deep_pct',
    'shortwave_radiation': 'solar_radiation_wm2',
    'evapotranspiration': 'evapotranspiration_mm',
    'vapour_pressure_deficit': 'vpd_kpa'
}, inplace=True)

print("Dataset expanded to 14 features and columns standardized.")
display(df.head())

Dataset expanded to 14 features and columns standardized.


,time,temperature_c,humidity_percent,precipitation_mm,dewpoint_c,pressure_hpa,cloud_cover_pct,wind_speed_kmh,wind_direction_deg,soil_temp_shallow_c,soil_moisture_shallow_pct,soil_moisture_deep_pct,solar_radiation_wm2,evapotranspiration_mm,vpd_kpa
0,2023-01-01 00:00:00,23.8,54,0.0,14.0,962.5,73,3.8,41,26.0,0.044,0.311,0.0,None,1.35
1,2023-01-01 01:00:00,22.8,54,0.0,13.0,962.0,98,9.8,73,25.1,0.044,0.311,0.0,None,1.27
2,2023-01-01 02:00:00,21.7,59,0.0,13.3,961.5,96,8.1,58,24.3,0.044,0.311,0.0,None,1.07
3,2023-01-01 03:00:00,21.5,60,0.0,13.5,961.5,96,2.5,315,23.4,0.044,0.311,0.0,None,1.01
4,2023-01-01 04:00:00,20.4,66,0.0,14.0,961.5,95,3.7,259,22.7,0.044,0.311,0.0,None,0.80


### 5. Handle Missing Values and Indexing: Drop NaNs and set the index
We will remove any rows containing null values to ensure data integrity and set the `time` column as our index, which is standard practice for time-series analysis.

In [7]:
# Ensure all numeric columns are correctly typed and handle potential string artifacts
for col in df.columns:
    if col != 'time':
        df[col] = pd.to_numeric(df[col], errors='coerce')

# Use Forward and Backward fill instead of just dropna to preserve temporal continuity
# Especially important for soil and radiation data
df_cleaned = df.ffill().bfill()

# Set the 'time' column as the index
df_cleaned.set_index('time', inplace=True)

print("--- Agricultural Dataset Cleaned ---")
print(f"Final column count: {len(df_cleaned.columns)}")
display(df_cleaned.head())
print("\nDataset Summary:")
print(df_cleaned.info())

--- Agricultural Dataset Cleaned ---
Final column count: 14


,temperature_c,humidity_percent,precipitation_mm,dewpoint_c,pressure_hpa,cloud_cover_pct,wind_speed_kmh,wind_direction_deg,soil_temp_shallow_c,soil_moisture_shallow_pct,soil_moisture_deep_pct,solar_radiation_wm2,evapotranspiration_mm,vpd_kpa
time,,,,,,,,,,,,,,
2023-01-01 00:00:00,23.8,54,0.0,14.0,962.5,73,3.8,41,26.0,0.044,0.311,0.0,NaN,1.35
2023-01-01 01:00:00,22.8,54,0.0,13.0,962.0,98,9.8,73,25.1,0.044,0.311,0.0,NaN,1.27
2023-01-01 02:00:00,21.7,59,0.0,13.3,961.5,96,8.1,58,24.3,0.044,0.311,0.0,NaN,1.07
2023-01-01 03:00:00,21.5,60,0.0,13.5,961.5,96,2.5,315,23.4,0.044,0.311,0.0,NaN,1.01
2023-01-01 04:00:00,20.4,66,0.0,14.0,961.5,95,3.7,259,22.7,0.044,0.311,0.0,NaN,0.80



Dataset Summary:
<class 'pandas.DataFrame'>
DatetimeIndex: 8760 entries, 2023-01-01 00:00:00 to 2023-12-31 23:00:00
Data columns (total 14 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   temperature_c              8760 non-null   float64
 1   humidity_percent           8760 non-null   int64  
 2   precipitation_mm           8760 non-null   float64
 3   dewpoint_c                 8760 non-null   float64
 4   pressure_hpa               8760 non-null   float64
 5   cloud_cover_pct            8760 non-null   int64  
 6   wind_speed_kmh             8760 non-null   float64
 7   wind_direction_deg         8760 non-null   int64  
 8   soil_temp_shallow_c        8760 non-null   float64
 9   soil_moisture_shallow_pct  8760 non-null   float64
 10  soil_moisture_deep_pct     8760 non-null   float64
 11  solar_radiation_wm2        8760 non-null   float64
 12  evapotranspiration_mm      0 non-null      float64
 13  vpd_k

### 6. Save the Data: Export as CSV
Finally, we export the cleaned DataFrame to a CSV file named `weather_cleaned.csv` for future use.

In [8]:
# Define the local path for cleaned data
interim_path = r'E:\Apollo_AgriVerse\02_Datasets\Interim\weather'
if not os.path.exists(interim_path):
    os.makedirs(interim_path, exist_ok=True)

# Save the cleaned data to CSV with the specified filename
# index=True preserves our time index
df_cleaned.to_csv(os.path.join(interim_path, "weather_clean.csv"), index=True)

print(f"Data successfully saved to: {interim_path}")

Data successfully saved to: E:\Apollo_AgriVerse\02_Datasets\Interim\weather


### 7. Feature Engineering (Time Components)
Extract useful patterns from the datetime index. Since machine learning models work best with numeric patterns, we extract the hour, day, and month. To make this even more effective, we apply cyclical encoding (Sine/Cosine) so the model understands that December (12) is next to January (1).

In [9]:
import numpy as np

# --- 1. Memory Downcasting (Updated for 14 columns) ---
float_cols = df_cleaned.select_dtypes(include=['float64']).columns
df_cleaned[float_cols] = df_cleaned[float_cols].astype('float32')

int_cols = df_cleaned.select_dtypes(include=['int64']).columns
df_cleaned[int_cols] = df_cleaned[int_cols].astype('int32')

# --- 2. Cyclical Feature Encoding ---
df_cleaned['hour'] = df_cleaned.index.hour
df_cleaned['month'] = df_cleaned.index.month

df_cleaned['hour_sin'] = np.sin(2 * np.pi * df_cleaned['hour'] / 24.0)
df_cleaned['hour_cos'] = np.cos(2 * np.pi * df_cleaned['hour'] / 24.0)
df_cleaned['month_sin'] = np.sin(2 * np.pi * df_cleaned['month'] / 12.0)
df_cleaned['month_cos'] = np.cos(2 * np.pi * df_cleaned['month'] / 12.0)

print(f"Memory optimized. New column list:\n{df_cleaned.columns.tolist()}")

Memory optimized. New column list:
['temperature_c', 'humidity_percent', 'precipitation_mm', 'dewpoint_c', 'pressure_hpa', 'cloud_cover_pct', 'wind_speed_kmh', 'wind_direction_deg', 'soil_temp_shallow_c', 'soil_moisture_shallow_pct', 'soil_moisture_deep_pct', 'solar_radiation_wm2', 'evapotranspiration_mm', 'vpd_kpa', 'hour', 'month', 'hour_sin', 'hour_cos', 'month_sin', 'month_cos']


### 8. Create Lag and Rolling Features
To predict future weather, the model needs past context. We create "lag features" (values from 1-3 hours ago) and "rolling averages" to capture short-term trends.

In [10]:
# --- 3. Rolling Window Statistics (Expanded for Agriculture) ---
# Using min_periods=1 to prevent dropping the first few rows

# Ensure the necessary columns exist before creating rolling/lag features
available_cols = df_cleaned.columns

# 24-hour rolling averages
if 'temperature_c' in available_cols:
    df_cleaned['temp_roll_avg_24h'] = df_cleaned['temperature_c'].rolling(window=24, min_periods=1).mean()
if 'humidity_percent' in available_cols:
    df_cleaned['humidity_roll_avg_24h'] = df_cleaned['humidity_percent'].rolling(window=24, min_periods=1).mean()
if 'solar_radiation_wm2' in available_cols:
    df_cleaned['solar_roll_avg_24h'] = df_cleaned['solar_radiation_wm2'].rolling(window=24, min_periods=1).mean()
if 'soil_moisture_shallow_pct' in available_cols:
    df_cleaned['soil_moist_roll_avg_24h'] = df_cleaned['soil_moisture_shallow_pct'].rolling(window=24, min_periods=1).mean()

# 24-hour volatility (standard deviation)
if 'temperature_c' in available_cols:
    df_cleaned['temp_volatility_24h'] = df_cleaned['temperature_c'].rolling(window=24, min_periods=2).std().fillna(0)
if 'vpd_kpa' in available_cols:
    df_cleaned['vpd_volatility_24h'] = df_cleaned['vpd_kpa'].rolling(window=24, min_periods=2).std().fillna(0)

# Lag features for immediate past context (1-hour shift)
if 'temperature_c' in available_cols:
    df_cleaned['temp_lag_1h'] = df_cleaned['temperature_c'].shift(1).bfill()
if 'precipitation_mm' in available_cols:
    df_cleaned['precip_lag_1h'] = df_cleaned['precipitation_mm'].shift(1).bfill()
if 'evapotranspiration_mm' in available_cols:
    df_cleaned['evapo_lag_1h'] = df_cleaned['evapotranspiration_mm'].shift(1).bfill()

print("Rolling statistics and lag features created for 14-variable agricultural dataset!")
# Display preview of agricultural engineered features
preview_cols = [c for c in ['temperature_c', 'temp_roll_avg_24h', 'solar_roll_avg_24h', 'soil_moist_roll_avg_24h', 'vpd_volatility_24h'] if c in df_cleaned.columns]
display(df_cleaned[preview_cols].head())

Rolling statistics and lag features created for 14-variable agricultural dataset!


,temperature_c,temp_roll_avg_24h,solar_roll_avg_24h,soil_moist_roll_avg_24h,vpd_volatility_24h
time,,,,,
2023-01-01 00:00:00,23.799999,23.799999,0.0,0.044,0.000000
2023-01-01 01:00:00,22.799999,23.299999,0.0,0.044,0.056569
2023-01-01 02:00:00,21.700001,22.766666,0.0,0.044,0.144222
2023-01-01 03:00:00,21.500000,22.450000,0.0,0.044,0.161142
2023-01-01 04:00:00,20.400000,22.040000,0.0,0.044,0.218174


### 9. Perform a Time-Aware Train/Test Split
In time-series, we must split data sequentially (`shuffle=False`). This ensures the model is trained on the past and tested on the future, preventing "data leakage" where the model accidentally sees future information during training.

In [11]:
from sklearn.model_selection import train_test_split

# Fix: Ensure all columns are numeric before splitting
# The previous error occurred because df_cleaned was empty (0 samples)
if 'evapotranspiration' in df_cleaned.columns:
    df_cleaned['evapotranspiration'] = pd.to_numeric(df_cleaned['evapotranspiration'], errors='coerce')

# Fill any specific remaining NaNs to prevent the dataframe from being empty
df_cleaned = df_cleaned.ffill().bfill()

# Define Target (y) and Features (X)
y = df_cleaned['temperature_c']
X = df_cleaned.drop(columns=['temperature_c'])

print(f"Total samples available for splitting: {len(X)}")

if len(X) > 0:
    # Split sequentially: first 80% for training, last 20% for testing
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    print(f"Training rows: {len(X_train)} | Testing rows: {len(X_test)}")
    print(f"Date Range Train: {X_train.index.min()} to {X_train.index.max()}")
    print(f"Date Range Test: {X_test.index.min()} to {X_test.index.max()}")
else:
    print("Error: Dataset is still empty. Please check the data fetching step.")

Total samples available for splitting: 8760
Training rows: 7008 | Testing rows: 1752
Date Range Train: 2023-01-01 00:00:00 to 2023-10-19 23:00:00
Date Range Test: 2023-10-20 00:00:00 to 2023-12-31 23:00:00


### 10. Feature Scaling
We use a `StandardScaler` to ensure features like humidity (0-100) and precipitation (often 0-5) are on the same numeric scale. We fit the scaler ONLY on training data to mimic a real-world scenario where future data is unknown.

In [12]:
from sklearn.preprocessing import RobustScaler

# --- 4. Outlier-Safe Scaling ---
# RobustScaler uses median and IQR, preventing extreme weather events from skewing the scale
scaler = RobustScaler()

# Fit only on training data to prevent data leakage
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Data scaled safely with RobustScaler. Outliers preserved!")
print(f"First row of scaled training features:\n{X_train_scaled[0]}")

Data scaled safely with RobustScaler. Outliers preserved!
First row of scaled training features:
[ 6.81818182e-02  0.00000000e+00 -3.00884892e-01  1.21276540e+00
  2.95165394e-01 -1.21518974e+00 -1.16666667e+00 -4.58852836e-01
  1.38248787e-02  1.23076719e-01 -2.41935484e-02             nan
 -1.80616736e-01 -1.00000000e+00 -8.00000000e-01 -4.32978028e-17
  7.07106781e-01  0.00000000e+00  1.00000000e+00 -8.49089642e-01
  1.44942649e-01 -4.97951505e+00  1.19760461e-02 -2.62924283e+00
 -1.67552040e+00 -5.71428571e-01  0.00000000e+00             nan]


e:\Apollo_AgriVerse\myenv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1215: RuntimeWarning: All-NaN slice encountered
  return fnb._ureduce(a, func=_nanmedian, keepdims=keepdims,
e:\Apollo_AgriVerse\myenv\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:1396: RuntimeWarning: All-NaN slice encountered
  return _nanquantile_unchecked(


### 11. Save Processed Data
Now we save the final, scaled feature matrices and targets to the 'Processed' directory. This step preserves the state of the data after all feature engineering and scaling is complete.

In [13]:
import numpy as np
import os

# Define the local path for processed data
processed_path = r'E:\Apollo_AgriVerse\02_Datasets\Processed\weather'
if not os.path.exists(processed_path):
    os.makedirs(processed_path, exist_ok=True)

# Convert scaled arrays back to DataFrames to preserve column names for CSV export
X_train_processed = pd.DataFrame(X_train_scaled, columns=X_train.columns, index=X_train.index)
X_test_processed = pd.DataFrame(X_test_scaled, columns=X_test.columns, index=X_test.index)

# Save the processed features and targets
X_train_processed.to_csv(os.path.join(processed_path, "X_train_processed.csv"), index=True)
X_test_processed.to_csv(os.path.join(processed_path, "X_test_processed.csv"), index=True)
y_train.to_csv(os.path.join(processed_path, "y_train.csv"), index=True)
y_test.to_csv(os.path.join(processed_path, "y_test.csv"), index=True)

print(f"All processed data successfully saved to: {processed_path}")

All processed data successfully saved to: E:\Apollo_AgriVerse\02_Datasets\Processed\weather


### 12. Validating Zero Values
We will check the percentage of zero values in each column to ensure they align with physical expectations (e.g., radiation being zero at night).

In [14]:
import pandas as pd
import os

# Fallback for VS Code: Load the data if df_cleaned is not in memory
if 'df_cleaned' not in locals():
    path = r'E:\Apollo_AgriVerse\02_Datasets\Interim\weather\weather_clean.csv'
    if os.path.exists(path):
        print(f"Loading data from local path: {path}")
        df_cleaned = pd.read_csv(path, index_col='time', parse_dates=True)
    else:
        print("Error: df_cleaned not found in memory and local CSV is missing. Please run the previous cleaning cells first.")

if 'df_cleaned' in locals():
    # Calculate the count and percentage of zero values per column
    zero_counts = (df_cleaned == 0).sum()
    zero_percentage = (zero_counts / len(df_cleaned)) * 100

    zero_report = pd.DataFrame({
        'Zero Count': zero_counts,
        'Percentage (%)': zero_percentage
    }).sort_values(by='Percentage (%)', ascending=False)

    print("\nAnalysis of Zero Values in Dataset:")
    display(zero_report)

    # Quick logic check for Solar Radiation (if column exists)
    if 'solar_radiation_wm2' in df_cleaned.columns:
        night_hours = df_cleaned[df_cleaned.index.hour.isin([0, 1, 2, 3, 4, 21, 22, 23])]
        solar_zero_at_night = (night_hours['solar_radiation_wm2'] == 0).all()
        print(f"\nIs solar radiation correctly zero during late-night hours? {solar_zero_at_night}")


Analysis of Zero Values in Dataset:


,Zero Count,Percentage (%)
precipitation_mm,7741,88.367580
precip_lag_1h,7741,88.367580
solar_radiation_wm2,4035,46.061644
cloud_cover_pct,1894,21.621005
hour,365,4.166667
hour_sin,365,4.166667
solar_roll_avg_24h,7,0.079909
vpd_volatility_24h,1,0.011416
temp_volatility_24h,1,0.011416
dewpoint_c,1,0.011416



Is solar radiation correctly zero during late-night hours? True
